# Data Preparation and Cleaning

In this section, we will discuss the steps involved in preparing and cleaning the data for analysis. Data preparation is a crucial step in the data analysis process, as it ensures that the data is accurate, consistent, and ready for analysis.

In [314]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
df = pd.read_csv("../data/raw/online_retail_II.csv" , encoding="ISO-8859-1")
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/10 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/10 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/10 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/10 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/10 8:26,3.39,17850.0,United Kingdom


## Standarize columns and data types

We're going to standardize all the columns with the following steps:
- Remove spaces and lowercase all the column names
- Remove special characters from the column names
- Replace any remaining spaces with underscores

### Standarize column names

In [315]:
df.columns = df.columns.str.strip().str.lower().str.replace({
    ' ': '_',
    '(': '',
    ')': '',
    'invoicedate': 'invoice_date',
    'stockcode': 'stock_code',
})
df.columns

Index(['invoice', 'stock_code', 'description', 'quantity', 'invoice_date',
       'price', 'customer_id', 'country'],
      dtype='str')

### Convert data types

We'll also convert the data types of the columns to ensure that they are appropriate for analysis. This includes converting date columns to datetime format, and numeric columns to the appropriate numeric format.

In [316]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541910 entries, 0 to 541909
Data columns (total 8 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   invoice       541910 non-null  str    
 1   stock_code    541910 non-null  str    
 2   description   540456 non-null  str    
 3   quantity      541910 non-null  int64  
 4   invoice_date  541910 non-null  str    
 5   price         541910 non-null  float64
 6   customer_id   406830 non-null  float64
 7   country       541910 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 33.1 MB


In [317]:
df = df.dropna(subset=['customer_id'])
df['customer_id'] = df['customer_id'].astype('int64')
df['invoice_date'] = pd.to_datetime(df['invoice_date'] , format='%m/%d/%y %H:%M')
df['country'] = df['country'].astype('category')

In [318]:
df.head()

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom


## Handling Missing Values

We will handle missing values in the dataset. Missing values can lead to inaccurate analysis and should be addressed appropriately. We will replace missing Description values with "Unknown"

In [319]:
df['description'] = df['description'].fillna('Unknown')
df[df['description'] == 'Unknown']

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country


We'll find out that there's no NA values in the description column. It's because we drop the rows with NA values in the customer_id column, which also had NA values in the description column.

## Remove invalid or non-relevant transactions

### Summary

- Drop duplicate rows based on unique transaction subsets.
- Remove zero or negative price records.
- Exclude non-cancellation negative quantity rows unless they are explicitly valid.

### What do we going to do?

Well, first of all, we will drop the duplicate rows based on the unique transaction subsets. Then, we will remove the zero or negative price records. Finally, we will exclude non-cancellation negative quantity rows unless they are explicitly valid.

### Drop duplicates rows based on unique transaction subsets

In [320]:
df.nunique()
print(f"Number of unique customers: {df['customer_id'].nunique()}")

Number of unique customers: 4372


In [321]:
df.drop_duplicates(inplace=True)

In [322]:
print(f"Number of unique customers after dropping duplicates: {df['customer_id'].nunique()}")

Number of unique customers after dropping duplicates: 4372


### Remove zero or negative price records

In [323]:
print(f"Number of rows with zero or negative price: {df[df['price'] <= 0].shape[0]}")

Number of rows with zero or negative price: 40


In [324]:
df = df[df['price'] > 0]

In [325]:
print(f"Number of rows with zero or negative price: {df[df['price'] <= 0].shape[0]}")

Number of rows with zero or negative price: 0


### Exclude non-cancellation negative quantity rows

In [326]:
print(f"Number of negative quantity rows (cancellations): {df[(df['quantity'] <= 0) & (~df['invoice'].str.startswith('C'))].shape[0]}")

Number of negative quantity rows (cancellations): 0


In [327]:
negative_quantity_rows = df[(df['quantity'] < 0) & (~df['invoice'].str.startswith('C'))]
print(f"Number of negative quantity rows (non-cancellations): {negative_quantity_rows.shape[0]}")

Number of negative quantity rows (non-cancellations): 0


## Insight

Why we don't have negative quantity rows? Because we have already dropped the rows with NA values in the customer_id column, which also had NA values in the description column. Therefore, we don't have any negative quantity rows in the dataset.

## Create business flags

### Summary

- Add is_cancellation flag based on Invoice prefix "C".
- Add special_stock_code flag for adjustments, discounts, postage, and samples.
- Add a valid_transaction flag for records retained in the analysis.

### What do we going to do?

We will create business flags to identify specific types of transactions in the dataset. This includes adding an is_cancellation flag based on the Invoice prefix "C", a special_stock_code flag for adjustments, discounts, postage, and samples, and a valid_transaction flag for records retained in the analysis.

In [328]:
df["is_cancellation"] = df["invoice"].str.startswith('C')

In [329]:
unique_special_stock_codes = list(df[(df["stock_code"].str.len() <= 4)]["stock_code"].unique())
df["special_stock_code"] = df["stock_code"].isin(unique_special_stock_codes)

In [330]:
df['valid_transaction'] = df["valid_transaction"] = (
    (~df["is_cancellation"]) &
    (~df["special_stock_code"]) &
    (df["quantity"] > 0) &
    (df["price"] > 0) &
    (df["customer_id"].notna())
)
print(f"Number of valid transactions: {df['valid_transaction'].sum()}")

Number of valid transactions: 391162


In [331]:
df.info()

<class 'pandas.DataFrame'>
Index: 401565 entries, 0 to 541909
Data columns (total 11 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   invoice             401565 non-null  str           
 1   stock_code          401565 non-null  str           
 2   description         401565 non-null  str           
 3   quantity            401565 non-null  int64         
 4   invoice_date        401565 non-null  datetime64[us]
 5   price               401565 non-null  float64       
 6   customer_id         401565 non-null  int64         
 7   country             401565 non-null  category      
 8   is_cancellation     401565 non-null  bool          
 9   special_stock_code  401565 non-null  bool          
 10  valid_transaction   401565 non-null  bool          
dtypes: bool(3), category(1), datetime64[us](1), float64(1), int64(2), str(3)
memory usage: 26.0 MB


In [332]:
df.head()

,invoice,stock_code,description,quantity,invoice_date,price,customer_id,country,is_cancellation,special_stock_code,valid_transaction
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,False,False,True
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,False,False,True
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,False,False,True
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,False,False,True
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,False,False,True


## Normalize and enrich the dataset

### Summary

- Handle special cases such as discounts and postage consistently.
- Derive useful transaction-level features such as total sales, order value, and purchase recency indicators.
- Create new columns for analysis, such as total sales, order value, and purchase recency indicators.

### What do we going to do?

We will handle special cases such as discounts and postage consistently, derive useful transaction-level features such as total sales, order value, and purchase recency indicators, and create new columns for analysis, such as total sales, order value, and purchase recency indicators.

In [333]:
df["is_discounted"] = df["stock_code"] =='D'
df["total_price"] = np.where((df["quantity"] >0) & (df["price"] > 0), df["quantity"] * df["price"], 0)
df["total_price"] = df["total_price"].round(2)
df["last_purchase_date"] = df.groupby("customer_id")["invoice_date"].transform("max")
df["days_since_last_purchase"] = (df["last_purchase_date"].max() - df["last_purchase_date"]).dt.days

## Summary

We create new columns for analysis, such as total sales, last purchase date, and days since last purchase. These new columns will help us to better understand the customer behavior and make more informed decisions.

### New columns created:

- **is_discounted:** A flag indicating whether the transaction is a discount or not.
- **total_price:** The total price of the transaction, calculated as quantity * price.
- **last_purchase_date:** The date of the last purchase made by the customer.
- **days_since_last_purchase:** The number of days since the last purchase made by the customer.
- **special_stock_code:** A flag indicating whether the stock code is a special stock code or not.


## Build customer-level dataset

We'll build a customer-level dataset by aggregating the transaction-level data. This will allow us to analyze customer behavior and segment customers based on their purchasing patterns.

### New dataset created:

- **customer_id:** The unique identifier for each customer.
- **recency:** The number of days since the last purchase made by the customer.
- **frequency:** The number of transactions made by the customer.
- **monetary**: The total amount spent by the customer.
- **avg_order_value:** The average order value for the customer, calculated as monetary / frequency.
- **avg_quantity:** The average quantity purchased by the customer, calculated as total quantity / frequency.
- **discount_rate:** The discount rate for the customer, calculated as total discounted transactions / total transactions.


In [334]:
customers = df.groupby("customer_id").agg(
    recency=("days_since_last_purchase", "min"),
    frequency=("invoice", "nunique"),
    monetary=("total_price", "sum"),
    avg_order_value=("total_price", "mean"),
    avg_quantity=("quantity", "mean"),
    discount_rate=("is_discounted", "mean"),
).reset_index()


print(f"Number of unique customers: {customers['customer_id'].nunique()}")

customers.describe().T

Number of unique customers: 4371


,count,mean,std,min,25%,50%,75%,max
customer_id,4371.0,15300.145276,1722.310262,12346.0,13813.50000,15301.0000,16778.500000,18287.000000
recency,4371.0,91.064974,100.770046,0.0,16.00000,49.0000,142.000000,373.000000
frequency,4371.0,5.075726,9.332529,1.0,1.00000,3.0000,5.000000,248.000000
monetary,4371.0,2033.225095,8952.996328,0.0,302.22500,659.4600,1647.370000,280206.020000
avg_order_value,4371.0,52.498700,878.772998,0.0,11.60500,17.2475,24.174776,42118.125000
avg_quantity,4371.0,19.542136,98.105165,-144.0,5.49056,9.5000,14.016667,4300.000000
discount_rate,4371.0,0.000095,0.002147,0.0,0.00000,0.0000,0.000000,0.090909


## Save the prepared dataset

Finally, we will save the prepared dataset to a CSV file for further analysis. This will allow us to easily access and analyze the data in the future.

- **`prepared_customers.csv`**: Contains the prepared customer data.
- **`prepared_transactions.csv`**: Contains the prepared transaction data.

We will save the prepared customer data to a CSV file named "prepared_customers.csv" and the prepared transaction data to a CSV file named "prepared_transactions.csv". This will allow us to easily access and analyze the data in the future.

In [335]:
customers.info()

<class 'pandas.DataFrame'>
RangeIndex: 4371 entries, 0 to 4370
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   customer_id      4371 non-null   int64  
 1   recency          4371 non-null   int64  
 2   frequency        4371 non-null   int64  
 3   monetary         4371 non-null   float64
 4   avg_order_value  4371 non-null   float64
 5   avg_quantity     4371 non-null   float64
 6   discount_rate    4371 non-null   float64
dtypes: float64(4), int64(3)
memory usage: 239.2 KB


In [336]:
customers.to_csv("../data/processed/prepared_customers.csv", index=False)
df.to_csv("../data/processed/prepared_transactions.csv", index=False)

## Conclusion of the Data Preparation Phase

The data preparation and cleaning phase has successfully transformed the raw online retail dataset into a structured, clean, and analysis-ready format. Following the planned blueprint, the key steps executed are:

* **Column Standardization**: Column names were stripped, lowercased, and formatted with underscores for consistency (e.g., `invoice`, `stock_code`, `customer_id`).
* **Data Type Conversion**: Dates were successfully parsed into datetime objects (`invoice_date`), customer IDs converted to integers, and country identifiers optimized as categorical types.
* **Missing Value Management**: Rows lacking a `customer_id` were filtered out, ensuring robust customer-level calculations, while description fields were normalized.
* **Invalid Transaction Removal**: Duplicate records were dropped, zero and negative price anomalies were removed, and transactions were validated to ensure clean behavioral analytics.
* **Business Flag Creation**: Custom indicators such as `is_cancellation`, `special_stock_code`, `valid_transaction`, and `is_discounted` were successfully implemented to isolate core purchasing behavior.
* **Feature Engineering & Aggregation**: Transaction-level metrics like `total_price` and recency indicators (`days_since_last_purchase`) were derived, culminating in a final customer-level summary dataset containing RFM metrics (Recency, Frequency, Monetary) alongside average order values and quantity averages.
* **Exporting Prepared Data**: The cleaned outputs were securely exported as `prepared_customers.csv` and `prepared_transactions.csv` to drive upcoming exploratory data analysis (EDA) and clustering models.

---

## Next Step: Exploratory Data Analysis (EDA)

The next phase, Exploratory Data Analysis (EDA), will focus on uncovering deep insights, patterns, and distributions from the newly prepared datasets (`prepared_customers.csv` and `prepared_transactions.csv`). The proposed path is:

* **Univariate Analysis of Customer Metrics**
  * Visualize distributions of Recency, Frequency, and Monetary (RFM) values across the customer base.
  * Check for skewness, outliers, and typical customer spend ranges.
* **Temporal and Transaction Trends**
  * Analyze purchasing volume trends over months, days of the week, and hours of the day.
  * Evaluate seasonality or peak periods in sales.
* **Geographical and Product Insights**
  * Examine top-performing countries and their contribution to overall monetary value.
  * Identify top-selling stock codes and product descriptions.
* **Correlation and Relationship Analysis**
  * Explore correlations between customer order frequency, average basket size, and total spend.
  * Investigate patterns among discounted transactions or special stock codes.

This EDA stage will establish a strong visual and statistical foundation, enabling precise configuration for the upcoming customer segmentation models.